### Configuración e Imports

In [ ]:
# ============================================================
# NB_TRNSF_GoldVentas_DimCalendario
# Capa: Gold | Modelo: ventas | Tabla: d_calendario
# Incluye: Script TP + API Feriados Argentina + BONUS Open-Meteo
# Nota: d_calendario NO está en origin_bronze_silver.
#       Es generada por script provisto por los profesores.
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    FloatType, IntegerType
)
from delta.tables import DeltaTable
import requests
from datetime import date

# ============================================================


### 1. Configuración Dinámica

In [ ]:
# 1. CONFIGURACIÓN DINÁMICA — SIN hardcodeos (requisito TP)
# ============================================================
_ctx    = notebookutils.runtime.context
WS_ID   = _ctx['workspaceId']
_lh_map = {lh.displayName: lh.id for lh in notebookutils.lakehouse.list()}
ART_GOLD = _lh_map.get('lh_gold', '')

# FIX D01/D02: leer modelo y nombre_tabla desde origin_gold
# d_calendario es orden=1 en origin_gold (script de profesores)
_og_cal = spark.sql("""
    SELECT modelo, nombre_tabla
    FROM lh_config.dbo.origin_gold
    WHERE nombre_notebook = 'NB_TRNSF_GoldVentas_DimCalendario' AND activo = 1
    LIMIT 1
""").collect()
if not _og_cal:
    raise Exception("Config no encontrada para DimCalendario en origin_gold")
_modelo_cal = _og_cal[0]['modelo']       # 'ventas'
_tabla_cal  = _og_cal[0]['nombre_tabla'] # 'd_calendario'

tabla_cal     = f'lh_gold.{_modelo_cal}.{_tabla_cal}'
abfs_gold_cal = (
    f'abfss://{WS_ID}'
    f'@onelake.dfs.fabric.microsoft.com'
    f'/{ART_GOLD}/Tables/{_modelo_cal}/{_tabla_cal}'
)

print(f'✅ Configuración cargada desde origin_gold')
print(f'   WS_ID         : {WS_ID}')
print(f'   ART_GOLD      : {ART_GOLD}')
print(f'   modelo        : {_modelo_cal}')
print(f'   nombre_tabla  : {_tabla_cal}')
print(f'   tabla_cal     : {tabla_cal}')
print(f'   abfs_gold_cal : {abfs_gold_cal}')

# ============================================================


### 2. Crear Tabla Base (Script TP)

In [ ]:
# 2. CREAR TABLA BASE — Script provisto por los profesores
# DROP TABLE garantiza limpieza total de metastore y archivos
# FIX: CAST(NULL AS STRING) evita error VOID en nombre_feriado
# ============================================================
print('\n🏗️  Creando estructura base de d_calendario...')
spark.sql(f'DROP TABLE IF EXISTS {tabla_cal}')

spark.sql(f"""
    CREATE TABLE {tabla_cal}
    USING DELTA
    AS
    WITH dates AS (
        SELECT explode(
            sequence(
                to_date('2020-01-01'),
                last_day(make_date(year(current_date()), 12, 1)),
                interval 1 day
            )
        ) AS Fecha
    )
    SELECT
        Fecha,
        day(Fecha)                             AS Nro_Dia,
        CASE dayofweek(Fecha)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Lunes'
            WHEN 3 THEN 'Martes'
            WHEN 4 THEN 'Miércoles'
            WHEN 5 THEN 'Jueves'
            WHEN 6 THEN 'Viernes'
            WHEN 7 THEN 'Sábado'
        END                                    AS Dia,
        lpad(month(Fecha), 2, '0')             AS Nro_Mes,
        CASE month(Fecha)
            WHEN 1  THEN 'Enero'       WHEN 2  THEN 'Febrero'
            WHEN 3  THEN 'Marzo'       WHEN 4  THEN 'Abril'
            WHEN 5  THEN 'Mayo'        WHEN 6  THEN 'Junio'
            WHEN 7  THEN 'Julio'       WHEN 8  THEN 'Agosto'
            WHEN 9  THEN 'Septiembre'  WHEN 10 THEN 'Octubre'
            WHEN 11 THEN 'Noviembre'   WHEN 12 THEN 'Diciembre'
        END                                    AS Mes,
        CASE month(Fecha)
            WHEN 1  THEN 'Ene' WHEN 2  THEN 'Feb' WHEN 3  THEN 'Mar'
            WHEN 4  THEN 'Abr' WHEN 5  THEN 'May' WHEN 6  THEN 'Jun'
            WHEN 7  THEN 'Jul' WHEN 8  THEN 'Ago' WHEN 9  THEN 'Sep'
            WHEN 10 THEN 'Oct' WHEN 11 THEN 'Nov' WHEN 12 THEN 'Dic'
        END                                    AS MesCorto,
        year(Fecha)                            AS Year,
        concat('S', weekofyear(Fecha))         AS Semana,
        (year(Fecha)*10000 + month(Fecha)*100 + day(Fecha)) AS DateKeyOrder,
        CAST(date_trunc('week', Fecha) AS DATE) AS StartOfWeek,
        date_add(date_trunc('week', Fecha), 6) AS EndOfWeek,
        concat('S', weekofyear(Fecha))         AS Semana_Year,
        quarter(Fecha)                         AS QuarterNumber,
        concat('Q', quarter(Fecha))            AS Trimestre,
        concat(lpad(month(Fecha),2,'0'), '/', right(year(Fecha),2)) AS CodeMes,
        concat(year(Fecha), lpad(month(Fecha),2,'0'))               AS CodTiempo,
        concat(
            CASE month(Fecha)
                WHEN 1  THEN 'Ene' WHEN 2  THEN 'Feb' WHEN 3  THEN 'Mar'
                WHEN 4  THEN 'Abr' WHEN 5  THEN 'May' WHEN 6  THEN 'Jun'
                WHEN 7  THEN 'Jul' WHEN 8  THEN 'Ago' WHEN 9  THEN 'Sep'
                WHEN 10 THEN 'Oct' WHEN 11 THEN 'Nov' WHEN 12 THEN 'Dic'
            END, '-', year(Fecha)
        )                                      AS Mes_Year,
        FALSE                                  AS es_feriado,
        CAST(NULL AS STRING)                   AS nombre_feriado
    FROM dates
""")

total_base = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabla_cal}').collect()[0]['cnt']
print(f'✅ Tabla creada: {total_base:,} filas')

# ============================================================


### 3. API Feriados Argentina

In [ ]:
# 3. API FERIADOS ARGENTINA (OBLIGATORIO)
# Fuente: https://api.argentinadatos.com/v1/feriados/{año}
# Años: todos los presentes en d_calendario
# ============================================================
print('\n🇦🇷  Consultando API Feriados Argentina...')

anios = [
    r['Year']
    for r in spark.sql(f'SELECT DISTINCT Year FROM {tabla_cal} ORDER BY Year').collect()
]
print(f'   Años a consultar: {anios}')

feriados_list = []
for anio in anios:
    try:
        url  = f'https://api.argentinadatos.com/v1/feriados/{anio}'
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data_anio = resp.json()
            # FIX [F01]: contador local por año, no búsqueda en lista acumulada
            for item in data_anio:
                feriados_list.append({
                    'fecha': item.get('fecha'),
                    'nombre': item.get('nombre', 'Feriado')
                })
            print(f'   ✅ {anio}: {len(data_anio)} feriados')
        else:
            print(f'   ⚠️ {anio}: HTTP {resp.status_code} — sin datos')
    except Exception as e:
        print(f'   ❌ {anio}: {e}')

print(f'   Total feriados obtenidos: {len(feriados_list)}')

# ============================================================


### 4. Merge Feriados → d_calendario

In [ ]:
# 4. MERGE FERIADOS → d_calendario
# Schema con DateType directo — evita doble conversión
# ============================================================
if feriados_list:
    schema_feriados = StructType([
        StructField('Fecha_fer',      DateType(),   True),
        StructField('nombre_feriado', StringType(), True),
    ])
    rows_fer = [
        (
            __import__('datetime').date.fromisoformat(f['fecha']),
            f['nombre']
        )
        for f in feriados_list if f['fecha']
    ]
    df_feriados = spark.createDataFrame(rows_fer, schema=schema_feriados)
    print(f'   Filas a mergear: {df_feriados.count()}')

    target_fer = DeltaTable.forName(spark, tabla_cal)
    target_fer.alias('t') \
        .merge(df_feriados.alias('f'), 't.Fecha = f.Fecha_fer') \
        .whenMatchedUpdate(set={
            'es_feriado':     F.lit(True),
            'nombre_feriado': F.col('f.nombre_feriado')
        }) \
        .execute()

    total_feriados = spark.sql(
        f'SELECT COUNT(*) as cnt FROM {tabla_cal} WHERE es_feriado = TRUE'
    ).collect()[0]['cnt']
    print(f'✅ Feriados actualizados en d_calendario: {total_feriados}')
else:
    print('⚠️ Sin feriados — MERGE omitido')

# ============================================================


### 5. BONUS — Clima Open-Meteo

In [ ]:
# 5. BONUS — CLIMA OPEN-METEO
# Fuente: archive-api.open-meteo.com (histórico) / api.open-meteo.com (actual)
# Ciudad: Nueva York (lat 40.7128, lon -74.0060) — sede de AdventureWorks
# Variables: Temperatura_Max, Temperatura_Min, Precipitacion_mm,
#            Codigo_Clima, Desc_Clima, Viento_Max_kmh
# ============================================================
print('\n🌤️  BONUS: Cargando datos climáticos (Open-Meteo)...')

LAT        = 40.7128
LON        = -74.0060
CIUDAD     = 'Nueva York'
AÑO_INICIO = 2011   # cubre f_ventas 2011-2014 + d_calendario 2020-2026
año_actual = date.today().year   # FIX [P02]: variable separada para el condicional

# FIX [H07]: diccionario WMO completo (24 códigos)
# Fuente: https://open-meteo.com/en/docs/weather-code
WMO_CODES = {
    0: 'Despejado',
    1: 'Mayormente despejado', 2: 'Parcialmente nublado', 3: 'Nublado',
    45: 'Niebla', 48: 'Niebla con escarcha',
    51: 'Llovizna leve', 53: 'Llovizna moderada', 55: 'Llovizna densa',
    61: 'Lluvia leve', 63: 'Lluvia moderada', 65: 'Lluvia intensa',
    71: 'Nevada leve', 73: 'Nevada moderada', 75: 'Nevada intensa',
    77: 'Granizo',
    80: 'Chubascos leves', 81: 'Chubascos moderados', 82: 'Chubascos violentos',
    85: 'Chubascos de nieve leves', 86: 'Chubascos de nieve intensos',
    95: 'Tormenta', 96: 'Tormenta con granizo leve', 99: 'Tormenta con granizo intenso'
}

BASE_URL     = 'https://archive-api.open-meteo.com/v1/archive'
FORECAST_URL = 'https://api.open-meteo.com/v1/forecast'
# FIX [S02]: pedir temperatura_min y viento en la misma llamada
VARIABLES    = 'temperature_2m_max,temperature_2m_min,precipitation_sum,weathercode,windspeed_10m_max'

registros_clima = []
for año in range(AÑO_INICIO, año_actual + 1):
    fecha_ini = f'{año}-01-01'
    fecha_fin = f'{año}-12-31' if año < año_actual else str(date.today())
    # FIX [P02]: usa año_actual (calculado 1 vez) — no AÑO_FIN
    url = BASE_URL if año < año_actual else FORECAST_URL

    params = {
        'latitude'  : LAT,
        'longitude' : LON,
        'start_date': fecha_ini,
        'end_date'  : fecha_fin,
        'daily'     : VARIABLES,
        'timezone'  : 'America/New_York'
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data   = r.json().get('daily', {})
        fechas = data.get('time',                [])
        t_max  = data.get('temperature_2m_max',  [None]*len(fechas))
        t_min  = data.get('temperature_2m_min',  [None]*len(fechas))  # FIX [S02]
        prec   = data.get('precipitation_sum',   [None]*len(fechas))
        wmo    = data.get('weathercode',          [None]*len(fechas))
        viento = data.get('windspeed_10m_max',   [None]*len(fechas))  # FIX [S02]

        for i, f in enumerate(fechas):
            codigo = int(wmo[i]) if wmo[i] is not None else None
            registros_clima.append({
                'fecha'          : f,
                'temp_max'       : float(t_max[i])  if t_max[i]  is not None else None,
                'temp_min'       : float(t_min[i])  if t_min[i]  is not None else None,  # FIX [S02]
                'precip_mm'      : float(prec[i])   if prec[i]   is not None else None,
                'codigo_clima'   : codigo,
                'desc_clima'     : WMO_CODES.get(codigo, f'Código {codigo}') if codigo is not None else None,
                'viento_max_kmh' : float(viento[i]) if viento[i] is not None else None,  # FIX [S02]
            })
        print(f'   ✅ {año}: {len(fechas)} días')
    except Exception as e:
        print(f'   ⚠️  {año}: error → {str(e)[:80]}')

print(f'   Total registros clima: {len(registros_clima)}')

# ============================================================


### 6. Merge Clima → d_calendario

In [ ]:
# 6. MERGE CLIMA → d_calendario
# Schema completo: 7 campos incluyendo temp_min y viento
# ============================================================
if registros_clima:
    # FIX [S03]: schema con todos los campos
    schema_clima = StructType([
        StructField('fecha',          StringType(), True),
        StructField('temp_max',       FloatType(),  True),
        StructField('temp_min',       FloatType(),  True),
        StructField('precip_mm',      FloatType(),  True),
        StructField('codigo_clima',   IntegerType(),True),
        StructField('desc_clima',     StringType(), True),
        StructField('viento_max_kmh', FloatType(),  True),
    ])

    df_clima = spark.createDataFrame(registros_clima, schema=schema_clima) \
        .withColumn('Fecha', F.to_date(F.col('fecha'), 'yyyy-MM-dd')) \
        .select(
            F.col('Fecha'),
            F.col('temp_max').alias('Temperatura_Max'),
            F.col('temp_min').alias('Temperatura_Min'),       # FIX [S01]
            F.col('precip_mm').alias('Precipitacion_mm'),
            F.col('codigo_clima').alias('Codigo_Clima'),
            F.col('desc_clima').alias('Desc_Clima'),
            F.col('viento_max_kmh').alias('Viento_Max_kmh'),  # FIX [S01]
        )

    total_clima_df = df_clima.count()
    print(f'   DataFrame clima: {total_clima_df:,} filas')

    # FIX [S01]: ALTER TABLE con las 6 columnas correctas
    existing_cols = [c.lower() for c in spark.table(tabla_cal).columns]
    if 'temperatura_max' not in existing_cols:
        spark.sql(f"""
            ALTER TABLE {tabla_cal}
            ADD COLUMNS (
                Temperatura_Max   FLOAT,
                Temperatura_Min   FLOAT,
                Precipitacion_mm  FLOAT,
                Codigo_Clima      INT,
                Desc_Clima        STRING,
                Viento_Max_kmh    FLOAT
            )
        """)
        print('   🛠️  Columnas climáticas agregadas al schema')

    df_clima.createOrReplaceTempView('vw_clima')

    target_clima = DeltaTable.forName(spark, tabla_cal)
    target_clima.alias('t') \
        .merge(df_clima.alias('c'), 't.Fecha = c.Fecha') \
        .whenMatchedUpdate(set={
            'Temperatura_Max'  : F.col('c.Temperatura_Max'),
            'Temperatura_Min'  : F.col('c.Temperatura_Min'),   # FIX [S01]
            'Precipitacion_mm' : F.col('c.Precipitacion_mm'),
            'Codigo_Clima'     : F.col('c.Codigo_Clima'),
            'Desc_Clima'       : F.col('c.Desc_Clima'),
            'Viento_Max_kmh'   : F.col('c.Viento_Max_kmh'),    # FIX [S01]
        }) \
        .execute()

    con_clima = spark.sql(
        f'SELECT COUNT(*) as cnt FROM {tabla_cal} WHERE Temperatura_Max IS NOT NULL'
    ).collect()[0]['cnt']
    print(f'✅ MERGE clima ejecutado. Cobertura: {con_clima:,} filas')
else:
    print('⚠️ Sin datos climáticos — MERGE clima omitido')

# ============================================================


### 7. Verificación Final

In [ ]:
# 7. VERIFICACIÓN FINAL
# ============================================================
print('\n📊 Verificación Final:')

display(spark.sql(f"""
    SELECT
        COUNT(*)                                                   AS Total_Filas,
        SUM(CASE WHEN es_feriado = TRUE THEN 1 ELSE 0 END)        AS Cantidad_Feriados,
        SUM(CASE WHEN Temperatura_Max IS NOT NULL THEN 1 ELSE 0 END) AS Dias_Con_Clima
    FROM {tabla_cal}
"""))

display(spark.sql(f"""
    SELECT
        Fecha, Dia, Mes, Year,
        es_feriado, nombre_feriado,
        Temperatura_Max, Temperatura_Min,
        Precipitacion_mm, Desc_Clima, Viento_Max_kmh
    FROM {tabla_cal}
    WHERE es_feriado = TRUE OR Temperatura_Max IS NOT NULL
    ORDER BY Fecha DESC
    LIMIT 20
"""))
